# MetaGPT (Model-Exclusive Task Arithmetic) Merge (FP32)

This notebook merges IF and Math checkpoints with MetaGPT closed-form scaling,
without using additional data.

- Task-vector definition (same style as `01_layer_interference_diagnostics.ipynb`):
  \[\Delta_t = 	heta_t - 	heta_0\]
- MetaGPT closed-form scaling:
  \[\lambda_t = rac{\|	heta_t - 	heta_0\|^2}{\sum_{k=1}^{n}\|	heta_k - 	heta_0\|^2}\]
- Merge rule:
  \[	heta_{	ext{merge}} = 	heta_0 + \sum_t \lambda_t \cdot \Delta_t\]

All model load, merge, and save steps are enforced in FP32.

In [ ]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Mapping

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class TaskSpec:
    """Task checkpoint specification used in MetaGPT merge.

    Args:
        name: Task identifier used in metadata and output naming.
        model_path: Local checkpoint path for the task model.
    """

    name: str
    model_path: Path


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for FP32 MetaGPT merge.

    Args:
        base_model_id: Base model identifier/path used as the merge anchor.
        output_root: Root directory where merged artifacts are saved.
        seed: Random seed for reproducibility.
        model_dtype: Torch dtype used for load/merge/save. Must be float32.
        device: Runtime device string for merge execution.
    """

    base_model_id: str
    output_root: Path
    seed: int
    model_dtype: torch.dtype
    device: str


# -----------------------------------------------------------------------------
# Experiment configuration
# -----------------------------------------------------------------------------
# Keep paths aligned with notebook 01 so task vectors are built from the same
# model checkpoints and anchor definition.
BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
)
MATH_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"
)

TASK_SPECS: List[TaskSpec] = [
    TaskSpec(name="if", model_path=IF_MODEL_PATH),
    TaskSpec(name="math", model_path=MATH_MODEL_PATH),
]

RUNTIME = RuntimeConfig(
    base_model_id=BASE_MODEL_ID,
    output_root=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-metagpt-task-arithmetic-fp32"
    ),
    seed=42,
    model_dtype=torch.float32,
    device="cpu",
)

# Fail early so missing local checkpoints are reported with explicit paths.
for required_path in [task_spec.model_path for task_spec in TASK_SPECS]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required task checkpoint does not exist: {required_path}")

# The user explicitly requested FP32 end-to-end.
if RUNTIME.model_dtype != torch.float32:
    raise ValueError(
        "This notebook requires FP32 for model load/merge/save. "
        f"Configured dtype={RUNTIME.model_dtype}"
    )

RUNTIME.output_root.mkdir(parents=True, exist_ok=True)
(RUNTIME.output_root / "metadata").mkdir(parents=True, exist_ok=True)

print(f"Base model id: {RUNTIME.base_model_id}")
print(f"Task checkpoints: {[str(task.model_path) for task in TASK_SPECS]}")
print(f"Runtime dtype (must be FP32): {RUNTIME.model_dtype}")
print(f"Output root: {RUNTIME.output_root}")

In [ ]:
def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility.

    Args:
        seed: Integer random seed.

    Returns:
        None. Global RNG states are updated in-place.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    """Return current UTC timestamp for metadata versioning.

    Returns:
        ISO-8601 timestamp string in UTC.
    """

    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Persist JSON payload to disk with pretty formatting.

    Args:
        payload: JSON-serializable mapping object.
        output_path: Destination JSON path.

    Returns:
        None. File is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def format_float_token(value: float) -> str:
    """Convert a float to a filesystem-safe token.

    Args:
        value: Float value to encode.

    Returns:
        String token where `.` is replaced by `p` and `-` by `m`.
    """

    formatted = f"{float(value):.6f}".rstrip("0").rstrip(".")
    if formatted in {"", "-0"}:
        formatted = "0"
    return formatted.replace("-", "m").replace(".", "p")


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional `fix_mistral_regex=True` compatibility.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.

    Returns:
        Loaded tokenizer instance.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm_fp32(
    model_name_or_path: str | Path,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load CausalLM + tokenizer in strict FP32.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        device: Runtime device string.

    Returns:
        Tuple `(model, tokenizer)` loaded in eval mode.
    """

    resolved_path = str(model_name_or_path)

    # `torch_dtype=torch.float32` keeps loading explicit and reproducible.
    model = AutoModelForCausalLM.from_pretrained(
        resolved_path,
        torch_dtype=torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    # Cast again to guarantee all floating tensors are FP32 before arithmetic.
    model.to(device=device, dtype=torch.float32)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved_path)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def assert_model_float32(model: AutoModelForCausalLM, model_label: str) -> None:
    """Assert that all floating parameters in a model are FP32.

    Args:
        model: Model instance to validate.
        model_label: Readable label for error diagnostics.

    Returns:
        None. Raises ValueError if any floating parameter is not float32.
    """

    for parameter_name, parameter in model.named_parameters():
        if torch.is_floating_point(parameter.data) and parameter.data.dtype != torch.float32:
            raise ValueError(
                "Detected non-FP32 parameter despite strict FP32 requirement | "
                f"model={model_label} | parameter={parameter_name} | dtype={parameter.data.dtype}"
            )


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
    task_name: str,
) -> None:
    """Validate parameter-key and shape compatibility between models.

    Args:
        base_model: Base checkpoint model.
        task_model: Task checkpoint model.
        task_name: Task key used in diagnostics.

    Returns:
        None. Raises ValueError when mismatch is detected.
    """

    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    if set(base_params.keys()) != set(task_params.keys()):
        missing_in_task = sorted(set(base_params.keys()) - set(task_params.keys()))
        missing_in_base = sorted(set(task_params.keys()) - set(base_params.keys()))
        raise ValueError(
            "Named parameter keys mismatch across base/task models. "
            f"task={task_name}, missing_in_task={missing_in_task[:5]}, missing_in_base={missing_in_base[:5]}"
        )

    for parameter_name, base_parameter in base_params.items():
        if base_parameter.shape != task_params[parameter_name].shape:
            raise ValueError(
                "Parameter shape mismatch across base/task models. "
                f"task={task_name}, parameter={parameter_name}, "
                f"base_shape={tuple(base_parameter.shape)}, "
                f"task_shape={tuple(task_params[parameter_name].shape)}"
            )


def build_task_vector_and_norm_sq_notebook01_style(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
    task_name: str,
) -> tuple[Dict[str, torch.Tensor], Dict[str, float]]:
    """Build task vector and squared L2 norm using notebook-01 style delta logic.

    The delta computation intentionally mirrors `01_layer_interference_diagnostics.ipynb`:
    - `base_fp32 = base_param.detach().to(torch.float32)`
    - `delta = task_param.detach().to(torch.float32) - base_fp32`

    Args:
        base_model: Base model used as merge anchor (`theta_0`).
        task_model: Task model used to compute `theta_t - theta_0`.
        task_name: Task key used in progress labels.

    Returns:
        Tuple of:
        - task vector mapping `parameter_name -> delta tensor (CPU FP32)`
        - metric dictionary with `l2_norm_sq`, `l2_norm`, and `numel`
    """

    validate_parameter_compatibility(
        base_model=base_model,
        task_model=task_model,
        task_name=task_name,
    )

    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    task_vector: Dict[str, torch.Tensor] = {}
    l2_norm_sq = 0.0
    total_numel = 0

    with torch.no_grad():
        for parameter_name in tqdm(base_params.keys(), desc=f"Build task vector ({task_name})"):
            base_tensor = base_params[parameter_name].detach()
            task_tensor = task_params[parameter_name].detach()

            if not torch.is_floating_point(base_tensor):
                continue

            # Keep FP32 tensors on CPU for deterministic and numerically stable arithmetic.
            base_fp32 = base_tensor.to(torch.float32).cpu()
            task_fp32 = task_tensor.to(torch.float32).cpu()

            # Notebook-01 style task vector: delta = theta_task - theta_base.
            delta = task_fp32 - base_fp32
            task_vector[parameter_name] = delta

            # MetaGPT lambda uses squared L2 norm of each task vector.
            l2_norm_sq += float(torch.sum(delta * delta).item())
            total_numel += int(delta.numel())

    return task_vector, {
        "l2_norm_sq": float(l2_norm_sq),
        "l2_norm": float(np.sqrt(max(l2_norm_sq, 0.0))),
        "numel": int(total_numel),
    }


def compute_metagpt_lambdas(task_vector_metrics: Mapping[str, Mapping[str, float]]) -> Dict[str, float]:
    """Compute MetaGPT closed-form scaling coefficients.

    Formula:
        `lambda_t = ||theta_t - theta_0||^2 / sum_k ||theta_k - theta_0||^2`

    Args:
        task_vector_metrics: Mapping from task name to metric dictionary that
            includes `l2_norm_sq`.

    Returns:
        Task-keyed lambda mapping.
    """

    if not task_vector_metrics:
        raise ValueError("task_vector_metrics must not be empty.")

    norm_sq_by_task: Dict[str, float] = {}
    for task_name, metric_dict in task_vector_metrics.items():
        if "l2_norm_sq" not in metric_dict:
            raise KeyError(f"Missing 'l2_norm_sq' in task metrics for task={task_name}")
        norm_sq_by_task[task_name] = float(metric_dict["l2_norm_sq"])

    denominator = float(sum(norm_sq_by_task.values()))
    if denominator <= 0.0:
        raise ValueError(
            "Sum of task-vector squared norms is non-positive; cannot compute MetaGPT lambdas. "
            f"norm_sq_by_task={norm_sq_by_task}"
        )

    return {
        task_name: float(norm_sq / denominator)
        for task_name, norm_sq in norm_sq_by_task.items()
    }


def compute_pairwise_task_alignment(
    task_vectors: Mapping[str, Mapping[str, torch.Tensor]],
    task_vector_metrics: Mapping[str, Mapping[str, float]],
) -> Dict[str, Dict[str, float]]:
    """Compute pairwise task-vector dot/cosine diagnostics.

    Why this diagnostic is useful:
        MetaGPT's derivation assumes near-orthogonal task vectors. This helper
        reports empirical pairwise alignment so that assumption is visible in
        the saved metadata.

    Args:
        task_vectors: Mapping `task_name -> parameter_name -> delta tensor`.
        task_vector_metrics: Mapping with per-task `l2_norm_sq` values.

    Returns:
        Pairwise mapping keyed by `"taskA__taskB"` with `dot` and `cosine`.
    """

    task_names = sorted(task_vectors.keys())
    pairwise: Dict[str, Dict[str, float]] = {}

    for left_index in range(len(task_names)):
        for right_index in range(left_index + 1, len(task_names)):
            left_task = task_names[left_index]
            right_task = task_names[right_index]

            dot_value = 0.0
            left_vector = task_vectors[left_task]
            right_vector = task_vectors[right_task]

            # Both vectors were validated against the same base parameter set,
            # so key sets must be identical.
            if set(left_vector.keys()) != set(right_vector.keys()):
                raise ValueError(
                    "Task vector parameter keys mismatch during pairwise alignment. "
                    f"left={left_task}, right={right_task}"
                )

            for parameter_name in left_vector.keys():
                dot_value += float(torch.sum(left_vector[parameter_name] * right_vector[parameter_name]).item())

            left_norm_sq = float(task_vector_metrics[left_task]["l2_norm_sq"])
            right_norm_sq = float(task_vector_metrics[right_task]["l2_norm_sq"])
            denominator = max(np.sqrt(max(left_norm_sq, 0.0)) * np.sqrt(max(right_norm_sq, 0.0)), 1e-12)
            cosine_value = float(dot_value / denominator)

            pairwise[f"{left_task}__{right_task}"] = {
                "dot": float(dot_value),
                "cosine": float(max(min(cosine_value, 1.0), -1.0)),
            }

    return pairwise


def apply_task_arithmetic_inplace(
    base_model: AutoModelForCausalLM,
    task_vectors: Mapping[str, Mapping[str, torch.Tensor]],
    task_lambdas: Mapping[str, float],
) -> Dict[str, Any]:
    """Apply weighted task arithmetic in-place in FP32.

    Merge rule:
        `theta_merge = theta_base + sum_t(lambda_t * Delta_t)`

    Args:
        base_model: Base model to overwrite with merged parameters.
        task_vectors: Mapping `task_name -> parameter_name -> Delta tensor`.
        task_lambdas: Mapping `task_name -> lambda`.

    Returns:
        Summary dictionary with update counters and simple norm diagnostics.
    """

    vector_tasks = set(task_vectors.keys())
    lambda_tasks = set(task_lambdas.keys())
    if vector_tasks != lambda_tasks:
        raise ValueError(
            "Task mismatch between task_vectors and task_lambdas. "
            f"vector_tasks={sorted(vector_tasks)}, lambda_tasks={sorted(lambda_tasks)}"
        )

    base_params = dict(base_model.named_parameters())
    updated_parameter_count = 0
    skipped_non_floating_count = 0

    with torch.no_grad():
        for parameter_name, base_parameter in tqdm(base_params.items(), desc="Apply MetaGPT merge"):
            if not torch.is_floating_point(base_parameter.data):
                skipped_non_floating_count += 1
                continue

            base_fp32 = base_parameter.data.detach().to(torch.float32)
            merged_tensor = base_fp32.clone()

            # Add each task contribution with its MetaGPT coefficient.
            for task_name, task_vector in task_vectors.items():
                merged_tensor.add_(float(task_lambdas[task_name]) * task_vector[parameter_name].to(torch.float32))

            base_parameter.data.copy_(merged_tensor.to(torch.float32))
            updated_parameter_count += 1

    total_l1_norms: Dict[str, float] = {}
    for task_name, task_vector in task_vectors.items():
        # Store aggregate L1 norm as a cheap sanity metric for vector magnitude.
        total_l1_norms[task_name] = float(
            sum(float(tensor.abs().sum().item()) for tensor in task_vector.values())
        )

    return {
        "updated_parameter_count": int(updated_parameter_count),
        "skipped_non_floating_count": int(skipped_non_floating_count),
        "task_lambdas": {task: float(value) for task, value in task_lambdas.items()},
        "task_vector_total_l1_norm": total_l1_norms,
    }


def build_output_dir_name(task_lambdas: Mapping[str, float]) -> str:
    """Build artifact directory name that encodes MetaGPT lambda values.

    Args:
        task_lambdas: Mapping from task key to lambda scalar.

    Returns:
        Filesystem-safe directory name.
    """

    tokens: List[str] = []
    for task_name in sorted(task_lambdas.keys()):
        tokens.append(f"{task_name}_l{format_float_token(task_lambdas[task_name])}")
    return "metagpt_" + "_".join(tokens) + "_fp32"


def save_merged_artifacts(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save merged checkpoint artifacts and metadata.

    Args:
        model: Merged model in FP32.
        tokenizer: Tokenizer to save with checkpoint.
        output_dir: Destination directory.
        metadata: JSON-serializable merge metadata payload.

    Returns:
        None. Artifacts are written to disk.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    # Save model in safetensors format while preserving FP32 weights.
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(dict(metadata), output_dir / "merge_metadata.json")

In [ ]:
set_seed(RUNTIME.seed)

if RUNTIME.model_dtype != torch.float32:
    raise ValueError(
        "Runtime dtype changed unexpectedly. "
        f"Expected torch.float32, got {RUNTIME.model_dtype}"
    )

print("Loading base model in FP32...")
merge_base_model, merge_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(merge_base_model, model_label="base_model")

merge_task_vectors: Dict[str, Dict[str, torch.Tensor]] = {}
task_vector_metrics: Dict[str, Dict[str, float]] = {}
merge_task_model_paths: Dict[str, str] = {}

for task_spec in TASK_SPECS:
    print(f"Loading task model in FP32 | task={task_spec.name}")
    task_model, _ = load_causal_lm_fp32(
        model_name_or_path=task_spec.model_path,
        device=RUNTIME.device,
    )
    assert_model_float32(task_model, model_label=f"task_model:{task_spec.name}")

    task_vector, metrics = build_task_vector_and_norm_sq_notebook01_style(
        base_model=merge_base_model,
        task_model=task_model,
        task_name=task_spec.name,
    )
    merge_task_vectors[task_spec.name] = task_vector
    task_vector_metrics[task_spec.name] = metrics
    merge_task_model_paths[task_spec.name] = str(task_spec.model_path)

    print(
        f"Task vector stats | task={task_spec.name} | "
        f"l2_norm_sq={metrics['l2_norm_sq']:.6e} | "
        f"l2_norm={metrics['l2_norm']:.6e} | "
        f"numel={int(metrics['numel'])}"
    )

    # Release task model as soon as vector extraction completes.
    del task_model
    gc.collect()

metagpt_lambdas = compute_metagpt_lambdas(task_vector_metrics=task_vector_metrics)
print(f"MetaGPT lambdas: {metagpt_lambdas}")

pairwise_alignment = compute_pairwise_task_alignment(
    task_vectors=merge_task_vectors,
    task_vector_metrics=task_vector_metrics,
)
if pairwise_alignment:
    print(f"Pairwise task-vector alignment: {pairwise_alignment}")

merge_summary = apply_task_arithmetic_inplace(
    base_model=merge_base_model,
    task_vectors=merge_task_vectors,
    task_lambdas=metagpt_lambdas,
)

# Re-check dtype before saving so FP32 guarantee holds for output checkpoint.
assert_model_float32(merge_base_model, model_label="merged_model")

output_dir_name = build_output_dir_name(task_lambdas=metagpt_lambdas)
output_dir = RUNTIME.output_root / output_dir_name

metadata = {
    "created_at": now_iso(),
    "method": "metagpt_model_exclusive_task_arithmetic",
    "formulas": {
        "task_vector": "Delta_task = theta_task - theta_base",
        "lambda": "lambda_t = ||theta_t - theta_base||^2 / sum_k ||theta_k - theta_base||^2",
        "merge": "theta_merge = theta_base + sum_t(lambda_t * Delta_t)",
    },
    "assumptions": {
        "model_exclusive": True,
        "uses_additional_data": False,
        "task_vector_orthogonality_assumed": True,
        "ntk_linear_regime_assumed": True,
    },
    "runtime": {
        "base_model_id": RUNTIME.base_model_id,
        "output_root": str(RUNTIME.output_root),
        "seed": int(RUNTIME.seed),
        "model_dtype": str(RUNTIME.model_dtype),
        "merge_device": str(RUNTIME.device),
    },
    "task_models": merge_task_model_paths,
    "task_vector_metrics": task_vector_metrics,
    "metagpt_lambdas": {task: float(value) for task, value in metagpt_lambdas.items()},
    "pairwise_alignment": pairwise_alignment,
    "merge_summary": merge_summary,
}

save_merged_artifacts(
    model=merge_base_model,
    tokenizer=merge_tokenizer,
    output_dir=output_dir,
    metadata=metadata,
)

run_summary_path = RUNTIME.output_root / "metadata" / f"{output_dir_name}_run_summary.json"
save_json(metadata, run_summary_path)

print(f"Saved FP32 MetaGPT checkpoint: {output_dir}")
print(f"Saved run summary: {run_summary_path}")